In [1]:
import re
import emoji
import string
import requests
import spacy
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

nlp = spacy.load("pl_core_news_sm")

#Zadania - część 1
## tekst do przetworzenia w ćwiczeniach

taka_historia = '<p> Cześć!!! 👋 Witajcie na moim nowym blogu o sztucznej inteligencji...   Dzisiaj porozmawiamy o NLP (Natural Language Processing). </p>

Czy wiedzieliście, że aż 80% danych w firmach to dane nieustrukturyzowane??? 😲 Więcej informacji znajdziecie na stronie: https://www.przykladowastrona.pl/nlp-wstep lub pisząc na e-mail: kontakt@moj-blog-ai.com.pl.

#MachineLearning #DataScience @Kowalski_Data_Geek

W      niektórych    miejscach celowo zostawiłem  duuuuuużo spacji i tabulacji. ALBO NAPISAŁEM COŚ CAPSLOCKIEM, żebyście mieli co zmieniać na małe litery (tzw. lowercasing).
Warto usunąć z tego tekstu polskie stop-words, np.: "i", "w", "na", "oraz", "że".

Data publikacji: 12.05.2023 r., godz. 14:30. Zysk firmy wzrósł o $45,000 w Q3!
P.S. Nie zapomnijcie o usunięciu tagów HTML, np. <b>pogrubienia</b> i znaków interpunkcyjnych! 🚀'

### Zadanie 1
a) zamień wielkość znaków na znaki małe
b) pozbądź się nadmiarowych białych znaków
c) pozbądź się nadmiarowych znaków przestankowych następujących po sobie (!!!, ???). Możesz je znaleźć w module string.
d) zamień emotikony na ich tekstową postać
e) pozbądź się wszystkich tagów HTML

### Zadanie 2
Masz do dyspozycji trzy tagi:
- <NUM> - liczby, daty, wartości księgowe i wszystko co stanowi wartość numeryczną,
- <URL> - adresy URL
- <EMAIL> - adresy email
Za pomocą wyrażeń regularnych i/lub funkcji wbudowanych klasy str zamień wartości w tekście na powyższe tagi.

### Zadanie 3
Wykorzystując listę stop words z adresu https://github.com/bieli/stopwords/blob/master/polish.stopwords.txt pozbądź się wszystkich słów stop z tekstu.

In [2]:
brudny_tekst = """ '<p> Cześć!!! 👋 Witajcie na moim nowym blogu o sztucznej inteligencji...   Dzisiaj porozmawiamy o NLP (Natural Language Processing). </p>
"""

# --- ZADANIE 1 ---
def oczysc_tekst_zad1(tekst):
    tekst = tekst.lower() # a)
    tekst = re.sub(r'<[^>]+>', '', tekst) # e)
    tekst = emoji.demojize(tekst) # d)

    # c)
    for znak in string.punctuation:
        znak_escaped = re.escape(znak)
        tekst = re.sub(f'({znak_escaped}){{2,}}', r'\1', tekst)

    tekst = re.sub(r'\s+', ' ', tekst).strip() # b)
    return tekst

# --- ZADANIE 2 ---
def taguj_tekst_zad2(tekst):
    tekst = re.sub(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\(\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', '<URL>', tekst)
    tekst = re.sub(r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+', '<EMAIL>', tekst)
    tekst = re.sub(r'\b\d+\b', '<NUM>', tekst)
    return tekst

# --- ZADANIE 3 ---
url_stop = "https://raw.githubusercontent.com/bieli/stopwords/master/polish.stopwords.txt"
stop_words_pl = set(requests.get(url_stop).text.splitlines())

def usun_stopwords_zad3(tekst, stopwords):
    slowa = tekst.split()
    slowa_przefiltrowane = [slowo for slowo in slowa if slowo not in stopwords]
    return " ".join(slowa_przefiltrowane)


zadanie1 = oczysc_tekst_zad1(brudny_tekst)
zadanie2 = taguj_tekst_zad2(zadanie1)
zadanie3 = usun_stopwords_zad3(zadanie2, stop_words_pl)

print("Zadanie 1 (Podstawowe czyszczenie):", zadanie1)
print("Zadanie 2 (Tagowanie Regex):", zadanie2)
print("Zadanie 3 (Usunięcie StopWords):", zadanie3)

Zadanie 1 (Podstawowe czyszczenie): ' cześć! :waving_hand: witajcie na moim nowym blogu o sztucznej inteligencji. dzisiaj porozmawiamy o nlp (natural language processing).
Zadanie 2 (Tagowanie Regex): ' cześć! :waving_hand: witajcie na moim nowym blogu o sztucznej inteligencji. dzisiaj porozmawiamy o nlp (natural language processing).
Zadanie 3 (Usunięcie StopWords): ' cześć! :waving_hand: witajcie nowym blogu sztucznej inteligencji. porozmawiamy nlp (natural language processing).


# Część 2.1
### ZADANIE 2.1:
Napisz funkcję clean_text(raw_text), która przyjmie tekst, przetworzy go przez nlp(), a następnie zwróci string, z którego usunięto wszystkie tokeny będące:

znakami interpunkcyjnymi (token.is_punct),
liczbami (token.is_digit),
tzw. stop-wordami (token.is_stop).
Wskazówka: Złóż ze sobą zachowane tokeny z powrotem w jeden string oddzielony spacją (" ".join(...)). Przetestuj na zmiennej text.

### ZADANIE 2.2:
Rozbuduj swoją funkcję z Zadania 2.1 do nowej postaci preprocess_text(raw_text). Niech funkcja nadal usuwa znaki interpunkcyjne, liczby oraz stopwords, ALE zamiast oryginalnego słowa (token.text), niech dodaje do ostatecznego wyniku jego ujednolicony lemat (token.lemma_). Przekształć w ten sposób wszystkie lematy na małe litery (użyj .lower()).

Przetestuj wynik na napisie: "Szybkie psy goniły najszybsze koty po dachach budynków."

### ZADANIE 2.3:
Poniżej zadeklarowano mini-korpus recenzji. Twoim celem jest:

Nadpisanie każdej recenzji w liście (użyj pętli) za pomocą swojej funkcji z Zadania 2.2 (preprocess_text), by dokonać czyszczenia i lematyzacji.
Użycie TfidfVectorizer (zamiast CountVectorizer) na czystym korpusie.
Wyświetlenie nowej macierzy TF-IDF jako DataFrame Pandas (tak jak pokazano wyżej).
reviews = [
    "Telefon jest genialny, świetne zdjęcia robi.",
    "Nigdy więcej tego telefonu, zdjęcia są koszmarne!",
    "Telefon popsuł się po tygodniu, katastrofa i koszmar.",
    "Kocham te zdjęcia!"
]

In [3]:
# --- ZADANIE 2.1 ---
def clean_text(raw_text):
    doc = nlp(raw_text)
    zachowane_tokeny = [
        token.text for token in doc
        if not token.is_punct and not token.is_digit and not token.is_stop
    ]
    return " ".join(zachowane_tokeny)

# --- ZADANIE 2.2 ---
def preprocess_text(raw_text):
    doc = nlp(raw_text)
    zachowane_lematy = [
        token.lemma_.lower() for token in doc
        if not token.is_punct and not token.is_digit and not token.is_stop
    ]
    return " ".join(zachowane_lematy)

# Testy 2.1 i 2.2
zdanie = "Szybkie psy goniły najszybsze koty po dachach budynków."
print("Zadanie 2.1 ", clean_text(zdanie))
print("Zadanie 2.2 ", preprocess_text(zdanie))


# --- ZADANIE 2.3 ---
corpus = [
    "Bardzo lubię programować w języku Python.",
    "Python jest super językiem.",
    "Nie znoszę, kiedy mój kod nie działa."
]

korpus_oczyszczony = [preprocess_text(recenzja) for recenzja in corpus]
vectorizer = TfidfVectorizer()
macierz_tfidf = vectorizer.fit_transform(korpus_oczyszczony)

df_tfidf = pd.DataFrame(
    macierz_tfidf.toarray(),
    columns=vectorizer.get_feature_names_out()
)

print("\nZadanie 2.3 ")
display(df_tfidf)

Zadanie 2.1  Szybkie psy goniły najszybsze koty dachach budynków
Zadanie 2.2  szybki pies gonić najszyby kot dach budynek

Zadanie 2.3 


,działać,język,kod,lubić,programować,python,super,znoszieć
0,0.00000,0.428046,0.00000,0.562829,0.562829,0.428046,0.000000,0.00000
1,0.00000,0.517856,0.00000,0.000000,0.000000,0.517856,0.680919,0.00000
2,0.57735,0.000000,0.57735,0.000000,0.000000,0.000000,0.000000,0.57735


### Zadanie 3 - dla chętnych

Wykonaj oczyszczanie lektury "Calineczka", którą możesz pobrać uruchamiając kod z komórki poniżej. Zastanów się, które informacje z pliku są najistotniejsze w kontekście przygotowania korpusu do treningu modelu. Możesz wykorzystać również tokenizację i lematyzację analizując czy wszystkie słowa znalazły swoją formę podstawową. Opisz wnioski.

!curl https://wolnelektury.pl/media/book/txt/calineczka.txt > calineczka.txt

In [4]:
url_calineczka = "https://wolnelektury.pl/media/book/txt/calineczka.txt"
calineczka_surowa = requests.get(url_calineczka).text
calineczka_tekst = calineczka_surowa.split("-----")[0]
calineczka_doc = nlp(calineczka_tekst)

calineczka_lematy = [
    token.lemma_.lower() for token in calineczka_doc
    if not token.is_punct and not token.is_digit and not token.is_space and not token.is_stop
]
calineczka_czysta = " ".join(calineczka_lematy)

print("Fragment pierwotny:")
print(calineczka_tekst[:250], "\n")

print("Fragment oczyszczony:")
print(calineczka_czysta[:200], "...\n")

Fragment pierwotny:
Hans Christian Andersen

Calineczka
tłum. Cecylia Niewiadomska

ISBN 978-83-288-2047-0


Pewna dobra kobieta bardzo pragnęła mieć maleńkie dziecko, ale nie wiedziała, skąd by je wziąć. Poszła więc do czarownicy i rzekła:

— Tak bym chciała  

Fragment oczyszczony:
hans christian andersen calineczka tłum cecylia niewiadomska isbn pewna dobry kobieta pragnąć mieć maleńki dziecko wiedzieć wziąć poszła czarownica rzec chcieć mieć malutki dziecko powiedzieć robić że ...



### WNIOSKI
1. Usunięcie stopki to podstawa: Na końcu pobranego pliku była ogromna stopka z licencjami fundacji Wolne Lektury. Musiałam to uciąć (split), bo wektoryzator mógłby uznać słowa takie jak "fundacja" czy "ISBN" za słowa kluczowe w tej baśni, a to popsułoby cały model.
2. Znikające zaimki: Po wycięciu "stop words" tekst robi się bardzo krótki. Wyrzuciliśmy słowa takie jak "ona", "jej", "sobie", co oszczędza mnóstwo miejsca i wektorów. Ale szczerze mówiąc, przy takim opowiadaniu modelowi (np. sieci neuronowej) mogłoby być teraz ciężko zorientować się, kto w ogóle co robi.
3. Lematyzator nie ogarnia zdrobnień: Zauważyłam, że np. słowo "Calineczka" czy inne bajkowe, stare zdrobnienia nie zawsze ładnie wracają do podstawowej formy w tym modelu spacy. Narzędzia są uczone na współczesnych, normalnych tekstach (np. newsach), więc z baśniami trochę się gubią.